<img src="https://raw.githubusercontent.com/IDEALLab/EngiOpt/codex/dcc26-workshop-notebooks/workshops/dcc26/assets/engibench_logo.png" width="560"/>

# Notebook 01 — Training a generative model for inverse design

> **Colab users:** click **File ➜ Save a copy in Drive** before editing so your changes persist.

## Where we are in the workshop

In **Notebook 00** we met the classical optimizer (`problem.optimize`). For a *single* scenario it's hard to beat — it converges to a near-optimal beam. So why bring ML in?

Because classical optimization only solves **one scenario at a time, from scratch.** Each new load or volume budget is a fresh iterative solve — seconds to minutes. The moment you need designs for *many* scenarios — a parameter sweep, an interactive tool, the inner loop of a bigger optimization — that per-scenario cost is the bottleneck.

A generative model **amortizes** it: learn the `scenario → design` map *once* from the thousands of solves already sitting in the dataset, then answer any new scenario in a single forward pass.

A learned design isn't guaranteed optimal — so you use it where "instant and good" beats "slow and optimal," or **warm-start the optimizer with it** to reach the optimum in a fraction of the iterations.

This notebook builds that model.

## Install dependencies (Colab / fresh env only)

Skip this if your local environment already has `engibench` and `engiopt` installed.

In [ ]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # flip to True to force install locally

if IN_COLAB or FORCE_INSTALL:
    def _pip(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    _pip(["engibench[all]", "matplotlib", "tqdm"])
    _pip(["git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"])
    try:
        import torch  # noqa: F401
    except Exception:
        _pip(["torch", "torchvision"])
    print("Install complete.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install here.")

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch as th

from engibench.utils.all_problems import BUILTIN_PROBLEMS
from engiopt.cgan_cnn_2d.cgan_cnn_2d import Generator as CGAN2DGenerator
from engiopt.workshops.dcc26.notebook_helpers import (
    TrainingConfig,
    WorkshopGenerator,
    generate_designs,
    show_gen_vs_baseline,
    show_training_curve,
    show_training_progression,
    train_supervised_generator,
)

SEED = 7
random.seed(SEED); np.random.seed(SEED); th.manual_seed(SEED)

if th.cuda.is_available():
    DEVICE = th.device("cuda")
elif th.backends.mps.is_available():
    DEVICE = th.device("mps")
else:
    DEVICE = th.device("cpu")
print("Device:", DEVICE)

---
## 1 — *Load the problem — exactly the way Notebook 00 did*

Nothing new here. We ask for `beams2d` by name, and the problem object hands us everything we need: the design shape, the names of the conditions, and a train/val/test dataset of `(scenario, design)` pairs already pre-split for us.

Notice what we *didn't* have to do: write a data loader, define a schema, decide on a train/test split, normalize conditions, or figure out what a design even looks like. The benchmark already answered all of those.

In [ ]:
problem = BUILTIN_PROBLEMS["beams2d"](seed=SEED)
train_ds = problem.dataset["train"]
test_ds = problem.dataset["test"]

condition_keys = problem.conditions_keys
design_shape = problem.design_space.shape
n_conds = len(condition_keys)

print(f"Design shape   : {design_shape}")
print(f"Condition keys : {condition_keys}")
print(f"Train examples : {len(train_ds):,}")
print(f"Test examples  : {len(test_ds):,}")

---
## 2 — *The simplest ML model - linear regression*

Here we will build our first ML model from the `beams2d` training data. We want a linear model that takes in the operating conditions of the beam as input and outputs the optimal beam for those conditions. That is:

- `X` — the conditions for each design, plus a `1` bias column.
- `Y` — each 50×100 design flattened into 5000 numbers.

We wish to learn the ML model `W` with `X W ≈ Y`. The least-squares solution is closed-form — the **normal equations** `W = (XᵀX)⁻¹XᵀY` — so there's no gradient descent, just one `np.linalg.lstsq` call.

In [ ]:
# X = the conditions for each design, plus a column of 1s (the bias term).
cond_columns = [np.array(train_ds[k]) for k in condition_keys]
X_train = np.stack(cond_columns + [np.ones(len(train_ds))], axis=1).astype(np.float32)

# Y = each 50×100 design flattened into a single 5000-long row.
Y_train = np.array(train_ds["optimal_design"]).astype(np.float32).reshape(len(X_train), -1)

# Fit: solve  X · W ≈ Y  in closed form (the normal equations). One call, no training loop.
W, *_ = np.linalg.lstsq(X_train, Y_train, rcond=None)
print("X:", X_train.shape, " Y:", Y_train.shape, " W:", W.shape)

# Sanity check: does using the scenario beat ignoring it and always drawing the average beam?
with np.errstate(all="ignore"):  # silence a harmless NumPy matmul warning
    mse_fit = ((X_train @ W - Y_train) ** 2).mean()
mse_avg = ((Y_train.mean(axis=0) - Y_train) ** 2).mean()
print(f"pixel MSE — linear fit: {mse_fit:.4f}   vs   average beam: {mse_avg:.4f}")

It beats *"always predict the average beam,"* so the conditions carry real signal. But low MSE isn't the same as a good design (Notebook 02's whole point). Let's see what it actually draws.

In [ ]:
# Build the same feature matrix for held-out TEST scenarios the fit never saw, and apply W.
cond_columns = [np.array(test_ds[k]) for k in condition_keys]
X_test = np.stack(cond_columns + [np.ones(len(test_ds))], axis=1).astype(np.float32)

with np.errstate(all="ignore"):  # silence a harmless NumPy matmul warning
    pred = X_test @ W
pred = np.clip(pred, 0, 1).reshape(-1, *design_shape)  # rows back into 50×100 images, valid [0, 1]

# Compare to the optimizer's design for the same scenario (show_gen_vs_baseline draws the grid).
baseline = np.array(test_ds["optimal_design"]).astype(np.float32)
records = [{k: float(np.array(test_ds[k])[i]) for k in condition_keys} for i in range(6)]
show_gen_vs_baseline(pred[:6], baseline[:6], records, n_show=6)

### What the picture tells us

Top: linear prediction. Bottom: the optimizer's design.

The predictions are **blurry gray clouds**. They shift sensibly as the load moves, so the model learned *something* — but they aren't beams. This is the limit of a simple linear model.

---
## 3 — *Reshape the dataset into tensors the model can take*

The benchmark dataset already pairs each design with its scenario. All we need to do is stack the columns into NumPy arrays so PyTorch can batch them. Two lines of real work.

The only subtlety is that we rescale designs from `[0, 1]` (the physics convention — 0 = void, 1 = solid) to `[-1, 1]` (the neural-network convention — matches the `tanh` output of the generator we'll use below).

In [ ]:
# Conditions: one row per training design, one column per scenario field.
conds_np = np.stack([np.array(train_ds[k]) for k in condition_keys], axis=1).astype(np.float32)

# Designs are in [0, 1]; rescale to [-1, 1] to match the generator's tanh output.
designs_np = np.array(train_ds["optimal_design"]).astype(np.float32)
targets_np = designs_np * 2.0 - 1.0

print(f"conditions: {conds_np.shape},  range [{conds_np.min():.2f}, {conds_np.max():.2f}]")
print(f"targets   : {targets_np.shape}, range [{targets_np.min():.2f}, {targets_np.max():.2f}]")

---
## 4 — *Pick a generator off the EngiOpt shelf*

The nonlinear upgrade. EngiOpt ships conditional generators that honor the same contract — conditions in, design out — just with millions of parameters. We'll grab the CGAN-CNN generator and treat it as a black box: `(noise, conditions) → design`.

```
 noise ∈ ℝ^32 ───┐
                 ├── [ conditional CNN generator ] ──→ design ∈ [0, 1]^{50×100}
conditions ∈ ℝ^4 ┘
```

The new ingredient is `noise` — it's what lets the generator return *different* designs for one scenario, which linear regression never could.

`WorkshopGenerator` just reshapes so we can pass plain 2D tensors.

In [ ]:
LATENT_DIM = 32  # size of the random noise vector per sample

cnn_gen = CGAN2DGenerator(
    latent_dim=LATENT_DIM,
    n_conds=n_conds,
    design_shape=design_shape,
)
model = WorkshopGenerator(cnn_gen).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Generator ready: {n_params:,} parameters, input = noise({LATENT_DIM}) + conditions({n_conds})")

---
## 5 — *Train it, supervised, against the optimizer's answers*

The training loop is the simplest thing that could work: for each batch, feed random noise plus real scenarios to the generator and ask it to match the optimizer's design on that scenario (MSE loss). No adversarial training, no diffusion schedule — just supervised regression on a benchmark dataset.

This is deliberately a *weak* generative model. Real methods do better. We're using the simplest possible recipe because the point of the notebook is the *plumbing*, not the performance.

> **Heads-up:** ~1–2 minutes on a GPU, ~5–10 minutes on CPU. Bump `EPOCHS` later if you want sharper designs.

In [ ]:
EPOCHS = 10

# Watch the first 4 held-out test scenarios turn into designs as training proceeds.
watch_conds = np.stack([np.array(test_ds[k])[:4] for k in condition_keys], axis=1).astype(np.float32)
watch_baselines = np.array(test_ds["optimal_design"])[:4].astype(np.float32)

train_cfg = TrainingConfig(
    latent_dim=LATENT_DIM,
    epochs=EPOCHS,
    batch_size=64,
    lr=2e-4,
    device=DEVICE,
    snapshot_at_epochs=[1, EPOCHS // 2, EPOCHS],  # render the watch scenarios at these epochs
)

result = train_supervised_generator(
    model, conds_np, targets_np,
    config=train_cfg,
    snapshot_conditions=watch_conds,
)
losses = result["losses"]
snapshots = result["snapshots"]

print(f"Final loss after {EPOCHS} epochs: {losses[-1]:.5f}")

### Sanity-check the loss curve

In [ ]:
show_training_curve(losses)

### Watch the designs emerge

At epoch 1 the generator outputs essentially noise. Halfway through training you start seeing dark blobs where material should go. By the final epoch the shape roughly tracks the ground-truth beam in the bottom row — same conditions, drawn from the test set, never shown during training.

In [ ]:
show_training_progression(snapshots, baseline_designs=watch_baselines, n_show=4)

---
## 6 — *Use it to generate designs for unseen scenarios*

The whole premise was "train once, generate instantly." Now we test that workflow. We grab a batch of scenarios from the held-out test set — scenarios the model has never been trained on — and ask the generator for a design on each one.

No optimization loop. No FEM. Just a forward pass.

In [ ]:
N_SAMPLES = 24

# Grab a random batch of held-out test scenarios and generate a design for each.
rng = np.random.default_rng(SEED)
test_idx = rng.choice(len(test_ds), size=N_SAMPLES, replace=False)
test_conds_np = np.stack([np.array(test_ds[k])[test_idx] for k in condition_keys], axis=1).astype(np.float32)
baseline_designs = np.array(test_ds["optimal_design"])[test_idx].astype(np.float32)

gen_designs = generate_designs(model, test_conds_np, latent_dim=LATENT_DIM, device=DEVICE)

print(f"Generated {gen_designs.shape[0]} designs of shape {gen_designs.shape[1:]} in a single forward pass.")

### Generated vs. baseline — side by side

Top row: the generator's output. Bottom row: the optimizer's output for the same scenario. Both are attempts to answer the same question.

The comparison highlights three points:

- **Blurriness.** The generator hedges. MSE loss rewards the average of all plausible designs for a given scenario, and the average of two valid truss topologies is an invalid blur. That's a loss-function problem, not a benchmark problem.
- **Condition sensitivity.** Do the generated designs actually *change* when the scenario changes, or does the model output something close to "the dataset mean"? Eyeballing this is hard; measuring it is what Notebook 02 is for.
- **Plausibility.** Some of these look like beams. Some don't. A picture can't tell you whether the design is stiff under load, or whether the material budget is respected. *You need a simulator for that.*

In [ ]:
conditions_records = [
    {k: float(test_conds_np[i, j]) for j, k in enumerate(condition_keys)}
    for i in range(N_SAMPLES)
]

show_gen_vs_baseline(gen_designs, baseline_designs, conditions_records)

---
## 7 — *Export artifacts for Notebook 02*

Notebook 02 will load these three files and run the physics simulator on every generated design.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
ARTIFACT_DIR = (
    Path("/content/dcc26_artifacts") if IN_COLAB
    else Path("workshops/dcc26/artifacts")
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

np.save(ARTIFACT_DIR / "generated_designs.npy", gen_designs)
np.save(ARTIFACT_DIR / "baseline_designs.npy", baseline_designs)
with open(ARTIFACT_DIR / "conditions.json", "w") as f:
    json.dump(conditions_records, f, indent=2)

print(f"Saved artifacts to {ARTIFACT_DIR}:")
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(f"  {p.name}")

---
## Putting it together

Every box in the "generative model for inverse design" picture was filled by one of two sides — EngiBench or EngiOpt:

| What you need                              | Who provides it | One-line access |
|--------------------------------------------|-----------------|-----------------|
| The problem definition                     | EngiBench       | `BUILTIN_PROBLEMS["beams2d"]()` |
| `(scenario, design)` training pairs        | EngiBench       | `problem.dataset["train"]` |
| The shape and range of a valid design      | EngiBench       | `problem.design_space` |
| The scenario schema                        | EngiBench       | `problem.conditions_keys` |
| A conditional generator architecture       | EngiOpt         | `engiopt.cgan_cnn_2d.Generator` |
| A training recipe                          | EngiOpt         | `train_supervised_generator(...)` |



---
## Reflect before moving on

1. Look at the side-by-side figure. If you had to write a short paragraph in a paper claiming your generator "works," what evidence in that figure would you cite — and what would a skeptical reviewer push back on?
2. We trained the model to minimize pixel-MSE against the optimizer's designs. Name one thing a *lower* MSE could improve, and one thing a lower MSE says *nothing* about.
3. The generator runs in milliseconds; the optimizer takes seconds. At what break-even number of scenarios does "train a generator once" start paying off compared to just running the optimizer each time?

## Next

In **Notebook 02** we will discuss metrics for evaluating the models developed in this notebook. *That* is what lets us say whether the generator "works."